In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: Qedma의 Qiskit Function
*[API 참조](https://docs.quantum.ibm.com/api/functions/qedma-qesem)를 참조하세요*

> **Note:** Qiskit Functions는 IBM Quantum&reg; Premium Plan, Flex Plan, On-Prem(IBM Quantum Platform API 경유) Plan 사용자에게만 제공되는 실험적 기능입니다. 미리 보기 출시 상태에 있으며 변경될 수 있습니다.

## 개요
양자 처리 장치는 최근 몇 년간 크게 개선되었지만, 기존 하드웨어의 노이즈와 불완전성으로 인한 오류는 양자 알고리즘 개발자에게 여전히 중심적인 과제입니다. 고전적으로 검증할 수 없는 유틸리티 규모 양자 계산에 가까워짐에 따라, 보장된 정확도로 노이즈를 제거하는 솔루션이 점점 더 중요해지고 있습니다. 이 과제를 극복하기 위해 Qedma는 IBM Quantum Platform에 [Qiskit Function](/guides/functions)으로 원활하게 통합된 양자 오류 완화(QESEM)를 개발했습니다.

QESEM을 사용하면 사용자가 노이즈가 있는 QPU에서 양자 Circuit을 실행하여 근본적인 한계에 가까운 매우 효율적인 QPU 시간 오버헤드로 매우 정확한 오류 없는 결과를 얻을 수 있습니다. 이를 달성하기 위해 QESEM은 오류의 특성화 및 감소를 위해 Qedma가 개발한 독점 방법 모음을 활용합니다. 오류 감소 기법에는 게이트 최적화, 노이즈 인식 트랜스파일, 오류 억제(ES), 편향 없는 오류 완화(EM)가 포함됩니다. 이 특성화 기반 방법들의 조합으로 사용자는 일반적인 대용량 양자 Circuit에서 신뢰할 수 있고 오류 없는 결과를 얻을 수 있으며, 그렇지 않으면 달성할 수 없는 응용 프로그램을 실현할 수 있습니다.

기본 구성 요소에 대한 전체 설명과 유틸리티 규모 시연은 논문 [유틸리티 규모 양자 Circuit을 위한 신뢰할 수 있는 고정확도 오류 완화](https://arxiv.org/abs/2508.10997)를 참조하세요.
## 설명
Qedma의 QESEM 함수를 사용하여 오류 억제 및 완화로 Circuit을 쉽게 추정하고 실행하여 더 큰 Circuit 볼륨과 더 높은 정확도를 달성할 수 있습니다. QESEM을 사용하려면 양자 Circuit, 측정할 Observable 집합, 각 Observable에 대한 목표 통계 정확도, 선택한 QPU를 제공합니다. 목표 정확도로 Circuit을 실행하기 전에 Circuit 실행이 필요 없는 분석적 계산을 기반으로 필요한 QPU 시간을 추정할 수 있습니다. QPU 시간 추정에 만족하면 QESEM으로 Circuit을 실행할 수 있습니다.

Circuit을 실행하면 QESEM은 Circuit에 맞춤화된 장치 특성화 프로토콜을 실행하여 Circuit에서 발생하는 오류에 대한 신뢰할 수 있는 노이즈 모델을 생성합니다. 특성화를 기반으로 QESEM은 먼저 노이즈 인식 트랜스파일을 구현하여 입력 Circuit을 목표 Observable에 영향을 미치는 노이즈를 최소화하는 물리적 Qubit과 게이트 집합으로 매핑합니다. 여기에는 기본적으로 사용 가능한 게이트(IBM&reg; 장치의 CX/CZ)와 QESEM이 최적화한 추가 게이트를 포함하는 QESEM의 확장된 게이트 집합이 포함됩니다. 그런 다음 QESEM은 QPU에서 특성화 기반 ES 및 EM Circuit 집합을 실행하고 측정 결과를 수집합니다. 이 결과들은 고전적으로 후처리되어 요청된 정확도에 해당하는 각 Observable에 대한 편향 없는 기댓값과 오차 막대를 제공합니다.

![Qedma QESEM 개요](../docs/images/guides/qedma-qesem/overview.svg)
QESEM은 다양한 양자 응용 프로그램과 오늘날 달성 가능한 가장 큰 Circuit 볼륨에서 높은 정확도의 결과를 제공하는 것으로 입증되었습니다. QESEM은 아래 벤치마크 섹션에서 시연된 다음과 같은 사용자 지향 기능을 제공합니다:
-	**보장된 정확도:** QESEM은 Observable의 기댓값에 대한 편향 없는 추정값을 출력합니다. EM 방법은 이론적 보장을 갖추고 있으며, Qedma의 최첨단 특성화와 함께 완화가 사용자 지정 정확도까지 노이즈 없는 Circuit 출력으로 수렴하도록 보장합니다. 체계적인 오류나 편향이 발생하기 쉬운 많은 휴리스틱 EM 방법과 달리, QESEM의 보장된 정확도는 일반적인 양자 Circuit과 Observable에서 신뢰할 수 있는 결과를 보장하는 데 필수적입니다.
-	**대규모 QPU로의 확장성:** QESEM의 QPU 시간은 Circuit 볼륨에 따라 달라지지만, 그 외에는 qubit 수와 무관합니다. Qedma는 IBM Quantum 127 qubit Eagle 및 133 qubit Heron 장치를 포함하여 오늘날 사용 가능한 가장 큰 양자 장치에서 QESEM을 시연했습니다.
-	**응용 프로그램 무관:** QESEM은 해밀토니안 시뮬레이션, VQE, QAOA, 진폭 추정을 포함한 다양한 응용 프로그램에서 시연되었습니다. 사용자는 임의의 양자 Circuit과 측정할 Observable을 입력하고 정확한 오류 없는 결과를 얻을 수 있습니다. 유일한 제한은 접근 가능한 Circuit 볼륨과 출력 정확도를 결정하는 하드웨어 사양과 할당된 QPU 시간에 의해 결정됩니다. 대조적으로, 많은 오류 감소 솔루션은 응용 프로그램에 특화되거나 제어되지 않는 휴리스틱을 포함하여 일반적인 양자 Circuit과 응용 프로그램에 적용할 수 없습니다.
-  **확장된 게이트 집합:** QESEM은 분수 각도 게이트를 지원하고 IBM Quantum Heron 및 Eagle 장치에서 Qedma 최적화 분수 각도 $Rzz(\theta)$ 게이트를 제공합니다. 이 확장된 게이트 집합은 더 효율적인 컴파일을 가능하게 하고 기본 CX/CZ 컴파일에 비해 최대 2배 더 큰 Circuit 볼륨을 실현합니다.
-	**Multibase Observable:** QESEM은 일반 해밀토니안과 같이 많은 비교환 Pauli 문자열로 구성된 입력 Observable을 지원합니다. 측정 기저의 선택과 QPU 리소스 할당(샷 및 Circuit)의 최적화는 요청된 정확도를 위한 필요한 QPU 시간을 최소화하기 위해 QESEM에 의해 자동으로 수행됩니다. 하드웨어 충실도와 실행 속도를 고려하는 이 최적화를 통해 더 깊은 Circuit을 실행하고 더 높은 정확도를 얻을 수 있습니다.
## 벤치마크
QESEM은 다양한 사용 사례와 응용 프로그램에서 테스트되었습니다. 다음 예제는 QESEM으로 실행할 수 있는 워크로드 유형을 평가하는 데 도움이 됩니다.

주어진 Circuit과 Observable에 대한 오류 완화와 고전적 시뮬레이션의 어려움을 정량화하는 핵심 성과 지표는 **활성 볼륨**입니다: Circuit에서 Observable에 영향을 미치는 CNOT 게이트의 수. 활성 볼륨은 Circuit 깊이와 너비, Observable 가중치, Observable의 라이트콘을 결정하는 Circuit 구조에 따라 달라집니다. 자세한 내용은 [2024 IBM Quantum Summit](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s) 강연을 참조하세요. QESEM은 고볼륨 레짐에서 특히 큰 가치를 제공하며 일반적인 Circuit과 Observable에 대한 신뢰할 수 있는 결과를 제공합니다.

![활성 볼륨](../docs/images/guides/qedma-qesem/active_volume.svg)

| 응용 프로그램    | qubit 수 | 장치 | Circuit 설명 | 정확도 | 총 시간 | 런타임 사용 |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| VQE Circuit  | 8              | Eagle (r3) | 21개 총 레이어, 9개 측정 기저, 1D 체인                    | 98%      | 35분      | 14분         |
| Kicked Ising   | 28              | Eagle (r3) | 고유 레이어 3개 x 3단계, 2D heavy-hex 토폴로지                      | 97%     | 22분      | 4분          |
| Kicked Ising   | 28              | Eagle (r3) | 고유 레이어 3개 x 8단계, 2D heavy-hex 토폴로지                     | 97%      | 116분      | 23분          |
| 트로터화 해밀토니안 시뮬레이션   | 40  | Eagle (r3)            | 고유 레이어 2개 x 10 트로터 단계, 1D 체인                    | 97%      | 3시간     | 25분         |
| 트로터화 해밀토니안 시뮬레이션   | 119  | Eagle (r3)           | 고유 레이어 3개 x 9 트로터 단계, 2D heavy-hex 토폴로지                    | 95%      | 6.5시간     | 45분         |
| Kicked Ising  | 136             | Heron (r2) | 고유 레이어 3개 x 15단계, 2D heavy-hex 토폴로지                    | 99%      | 52분             | 9분           |

여기서 정확도는 Observable의 이상적인 값에 대한 상대적인 측정값입니다: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, 여기서 '$\epsilon$'은 완화의 절대 정밀도(사용자 입력으로 설정)이고, $\langle O \rangle_{ideal}$은 노이즈 없는 Circuit에서의 Observable 값입니다.
'런타임 사용'은 배치 모드에서의 벤치마크 사용량(개별 Job 사용량의 합계)을 측정하며, '총 시간'은 추가적인 고전 및 통신 시간을 포함하는 세션 모드에서의 사용량(실험 벽시계 시간)을 측정합니다. QESEM은 두 모드 모두에서 실행 가능하므로 사용자는 사용 가능한 리소스를 최대한 활용할 수 있습니다.

28 qubit Kicked Ising Circuit은 ibm_kawasaki의 세 개 연결 루프에서 Shinjo et al.이 연구한 이산 시간 준결정을 시뮬레이션합니다(참조 [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) 및 [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). 여기서 취한 Circuit 매개변수는 $(\theta_x, \theta_z) = (0.9 \pi, 0)$이며, 강자성 초기 상태 $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$입니다. 측정된 Observable은 자화의 절댓값 $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$입니다. 유틸리티 규모 Kicked Ising 실험은 ibm_fez의 136개 최고 Qubit에서 실행되었습니다. 이 특정 벤치마크는 Clifford 각도 $(\theta_x, \theta_z) = (\pi, 0)$에서 실행되었으며, 이 각도에서는 활성 볼륨이 Circuit 깊이에 따라 천천히 증가하여 높은 장치 충실도와 함께 짧은 런타임에 높은 정확도를 달성할 수 있습니다.

트로터화 해밀토니안 시뮬레이션 Circuit은 분수 각도의 횡방향 필드 이징 모델에 대한 것입니다: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ 및 $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$(참조 [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). 유틸리티 규모 Circuit은 ibm_brisbane의 119개 최고 Qubit에서 실행되었으며, 40 qubit 실험은 사용 가능한 최고의 체인에서 실행되었습니다. 정확도는 자화에 대해 보고됩니다. 더 높은 가중치의 Observable에 대해서도 높은 정확도의 결과를 얻었습니다.

VQE Circuit은 Deutsches Elektronen-Synchrotron(DESY)의 양자 기술 및 응용 센터 연구자들과 함께 개발되었습니다. 여기서 목표 Observable은 많은 수의 비교환 Pauli 문자열로 구성된 해밀토니안으로, 다중 기저 Observable에 대한 QESEM의 최적화된 성능을 강조합니다. 완화는 고전적으로 최적화된 앤자츠에 적용되었습니다. 이 결과들은 아직 미발표 상태이지만, 유사한 구조적 특성을 가진 다른 Circuit에서도 동일한 품질의 결과를 얻을 수 있습니다.
## 시작하기
[IBM Quantum Platform API 키](http://quantum.cloud.ibm.com/)를 사용하여 인증하고 다음과 같이 QESEM Qiskit Function을 선택하세요. (이 코드 스니펫은 이미 로컬 환경에 [계정을 저장](/guides/functions#install-qiskit-functions-catalog-client)했다고 가정합니다.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## 예제
시작하려면 주어진 `pub`에 대해 QESEM을 실행하는 데 필요한 QPU 시간을 추정하는 다음 기본 예제를 시도해 보세요:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

다음 예제는 QESEM Job을 실행합니다:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

익숙한 Qiskit Serverless API를 사용하여 Qiskit Function 워크로드의 상태를 확인하거나 결과를 반환할 수 있습니다:

In [ ]:
print(sample_job.status())
result = sample_job.result()

다음 코드 스니펫은 QPU 시간 추정값을 검색하는 방법을 설명합니다(`estimate_time_only`가 설정된 경우):

In [ ]:
print(
    f"The estimated QPU time for this PUB is: \n{time_estimate_result[0].metadata}"
)

다음 코드 스니펫은 완화 결과(`estimate_time_only`가 설정되지 않은 경우)와 실행 지표를 검색하는 방법을 보여줍니다. 이것들은 QESEM 실행에 다양한 매개변수가 미치는 영향에 대한 더 깊은 이해를 가능하게 하는 필수 데이터를 포함합니다. 연구를 기반으로 논문을 작성할 때도 관련이 있을 수 있습니다.

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## 오류 메시지 가져오기
워크로드 상태가 ERROR인 경우, `job.result()`를 사용하여 오류 메시지를 다음과 같이 가져오세요:

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com.

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
- Visit the [API reference](/docs/api/functions/qedma-qesem) for this Qiskit Function.
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).


</Admonition>